In [8]:
def is_variable(expr):
    """Returns True if the expression is a variable (lowercase string)."""
    return isinstance(expr, str) and expr.islower()

def is_constant(expr):
    """Returns True if the expression is a constant (uppercase string)."""
    return isinstance(expr, str) and not expr.islower()

def occurs_in(var, expr):
    """Checks if a variable occurs inside a compound expression (Occurs Check)."""
    if var == expr:
        return True
    if isinstance(expr, list):
        return any(occurs_in(var, sub_expr) for sub_expr in expr)
    return False

def apply_substitution(subst, expr):
    """Applies a substitution set to an expression recursively."""
    if not subst:
        return expr
    if isinstance(expr, str) and expr in subst:
        return subst[expr]
    if isinstance(expr, list):
        return [apply_substitution(subst, element) for element in expr]
    return expr

def unify(psi1, psi2):
    if isinstance(psi1, str) or isinstance(psi2, str):
        if psi1 == psi2:
            return {}
        elif is_variable(psi1):
            if occurs_in(psi1, psi2): return "FAILURE"
            else: return {psi1: psi2}
        elif is_variable(psi2):
            if occurs_in(psi2, psi1): return "FAILURE"
            else: return {psi2: psi1}
        else:
            return "FAILURE"

    if psi1[0] != psi2[0]:
        return "FAILURE"

    if len(psi1) != len(psi2):
        return "FAILURE"

    subst = {}

    for i in range(1, len(psi1)):
        s = unify(psi1[i], psi2[i])

        if s == "FAILURE":
            return "FAILURE"

        if s != {}:
            psi1 = apply_substitution(s, psi1)
            psi2 = apply_substitution(s, psi2)


            for k, v in subst.items():
                subst[k] = apply_substitution(s, v)
            subst.update(s)

    return subst


if __name__ == "__main__":
    print("--- Unification Algorithm (FOL) ---")
    print("Rules: Variables are lowercase ('x', 'y'). Constants/Predicates are uppercase ('A', 'P').")
    print("Nested Note: Nested predicates must use inner brackets, e.g., ['Father', 'y']")

    try:
        user_input1 = input("\nEnter Expression 1: ")
        user_input2 = input("Enter Expression 2: ")

        psi1 = eval(user_input1)
        psi2 = eval(user_input2)

        result = unify(psi1, psi2)

        print("\n--- Result ---")
        if result == "FAILURE":
            print("Unification Failed.")
        elif result == {}:
            print("Unification Successful: NIL (Expressions already identical).")
        else:
            print("Unification Successful!")
            print("Substitution Set (SUBST):")
            for var, val in result.items():
                print(f"  {var} / {val}")

    except Exception as e:
        print(f"\nInvalid input format: {e}")


--- Unification Algorithm (FOL) ---
Rules: Variables are lowercase ('x', 'y'). Constants/Predicates are uppercase ('A', 'P').
Nested Note: Nested predicates must use inner brackets, e.g., ['Father', 'y']

Enter Expression 1: ['Older',['Father', 'y'],'y']
Enter Expression 2: ['Older',['Father', 'x'],'John']

--- Result ---
Unification Successful!
Substitution Set (SUBST):
  y / John
  x / John
